In [1]:
from peft import LoraConfig

c:\PycharmProjects\projects\fine_tune_proj\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0622 15:53:24.773000 33436 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [6]:
# 1、构造LoRAConfig实例对象
lora_config = LoraConfig(
    r=8,
    lora_alpha=8,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj","v_proj"],
    task_type="CAUSAL_LM"
)

In [7]:
# 2、通过get_peft_model获取peftmodel
from transformers import AutoModelForCausalLM
model = AutoModelForCausalLM.from_pretrained("model/qwen3-0.6B")

Loading weights: 100%|██████████| 311/311 [00:00<00:00, 1303.26it/s, Materializing param=model.norm.weight]                              
The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [8]:
from peft import get_peft_model
peft_model = get_peft_model(model,lora_config)

In [9]:
# 查看当前模型的结构：新增了A,B可训练模块
peft_model

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3ForCausalLM(
      (model): Qwen3Model(
        (embed_tokens): Embedding(151936, 1024)
        (layers): ModuleList(
          (0-27): 28 x Qwen3DecoderLayer(
            (self_attn): Qwen3Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=1024, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1024, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Linear(in_fea

In [10]:
peft_model.print_trainable_parameters()

trainable params: 1,146,880 || all params: 752,779,264 || trainable%: 0.1524
